In [1]:
"""
Bag of Words (BoW) — Representação vetorial de texto
=====================================================
Conceito:
  Cada documento é representado como um vetor de contagens de palavras.
  A ordem das palavras é ignorada; apenas a frequência importa.

Etapas:
  1. Pré-processamento (lowercase, remoção de pontuação, stop words)
  2. Tokenização
  3. Construção do vocabulário
  4. Codificação dos documentos como vetores
  5. Visualização da matriz e análise de similaridade
"""

import numpy as np
import re
from collections import Counter

In [2]:
# ─────────────────────────────────────────────
# 1. CORPUS DE EXEMPLO
# ─────────────────────────────────────────────

corpus = [
    "O cachorro late para o gato",
    "O gato dorme na casa",
    "O cachorro corre na rua",
    "O gato e o cachorro brincam juntos",
    "A casa tem um cachorro e um gato",
]

In [3]:
# ─────────────────────────────────────────────
# 2. PRÉ-PROCESSAMENTO
# ─────────────────────────────────────────────

# Stop words: palavras muito comuns que carregam pouco significado
STOP_WORDS = {"o", "a", "os", "as", "um", "uma", "e", "na", "no", "para"}

In [4]:
def preprocessar(texto: str) -> list[str]:
    """
    Converte texto bruto em lista de tokens limpos.

    Passos:
      - lowercase
      - remoção de pontuação
      - tokenização por espaço
      - remoção de stop words
    """
    texto = texto.lower()
    texto = re.sub(r"[^\w\s]", "", texto)   # remove pontuação
    tokens = texto.split()
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return tokens

tokens_corpus = [preprocessar(doc) for doc in corpus]

print("=" * 55)
print("TOKENS POR DOCUMENTO")
print("=" * 55)
for i, (doc, tokens) in enumerate(zip(corpus, tokens_corpus)):
    print(f"\nDoc {i}: \"{doc}\"")
    print(f"  → {tokens}")


TOKENS POR DOCUMENTO

Doc 0: "O cachorro late para o gato"
  → ['cachorro', 'late', 'gato']

Doc 1: "O gato dorme na casa"
  → ['gato', 'dorme', 'casa']

Doc 2: "O cachorro corre na rua"
  → ['cachorro', 'corre', 'rua']

Doc 3: "O gato e o cachorro brincam juntos"
  → ['gato', 'cachorro', 'brincam', 'juntos']

Doc 4: "A casa tem um cachorro e um gato"
  → ['casa', 'tem', 'cachorro', 'gato']


In [5]:
# ─────────────────────────────────────────────
# 3. VOCABULÁRIO
# ─────────────────────────────────────────────

# Vocabulário: conjunto de todas as palavras únicas, ordenado
vocab = sorted(set(token for tokens in tokens_corpus for token in tokens))
word2idx = {palavra: idx for idx, palavra in enumerate(vocab)}
idx2word = {idx: palavra for palavra, idx in word2idx.items()}

print("\n" + "=" * 55)
print("VOCABULÁRIO")
print("=" * 55)
for palavra, idx in word2idx.items():
    print(f"  [{idx:2d}] {palavra}")

print(f"\nTamanho do vocabulário: {len(vocab)} palavras")



VOCABULÁRIO
  [ 0] brincam
  [ 1] cachorro
  [ 2] casa
  [ 3] corre
  [ 4] dorme
  [ 5] gato
  [ 6] juntos
  [ 7] late
  [ 8] rua
  [ 9] tem

Tamanho do vocabulário: 10 palavras


In [6]:
# ─────────────────────────────────────────────
# 4. CONSTRUÇÃO DA MATRIZ BOW
# ─────────────────────────────────────────────

def construir_bow(tokens_corpus: list[list[str]], word2idx: dict) -> np.ndarray:
    """
    Constrói a matriz Bag of Words.

    Retorna:
      np.ndarray de shape (n_documentos, tamanho_vocabulario)
      bow[i, j] = número de vezes que vocab[j] aparece no documento i
    """
    n_docs = len(tokens_corpus)
    n_vocab = len(word2idx)
    bow = np.zeros((n_docs, n_vocab), dtype=np.int32)

    for i, tokens in enumerate(tokens_corpus):
        contagem = Counter(tokens)
        for palavra, freq in contagem.items():
            if palavra in word2idx:
                bow[i, word2idx[palavra]] = freq

    return bow

bow = construir_bow(tokens_corpus, word2idx)

print("\n" + "=" * 55)
print("MATRIZ BAG OF WORDS")
print(f"Shape: {bow.shape}  (documentos × palavras)")
print("=" * 55)

# Imprime cabeçalho com as palavras
header = f"{'':>6}" + "".join(f"{p:>10}" for p in vocab)
print(header)
print("-" * len(header))

for i, vetor in enumerate(bow):
    linha = f"Doc {i}:" + "".join(f"{v:>10}" for v in vetor)
    print(linha)


MATRIZ BAG OF WORDS
Shape: (5, 10)  (documentos × palavras)
         brincam  cachorro      casa     corre     dorme      gato    juntos      late       rua       tem
----------------------------------------------------------------------------------------------------------
Doc 0:         0         1         0         0         0         1         0         1         0         0
Doc 1:         0         0         1         0         1         1         0         0         0         0
Doc 2:         0         1         0         1         0         0         0         0         1         0
Doc 3:         1         1         0         0         0         1         1         0         0         0
Doc 4:         0         1         1         0         0         1         0         0         0         1


In [7]:
# ─────────────────────────────────────────────
# 5. ANÁLISE: PALAVRAS MAIS FREQUENTES
# ─────────────────────────────────────────────

print("\n" + "=" * 55)
print("FREQUÊNCIA GLOBAL DAS PALAVRAS (todos os docs)")
print("=" * 55)

freq_global = bow.sum(axis=0)  # soma ao longo dos documentos
ranking = sorted(zip(vocab, freq_global), key=lambda x: -x[1])

for palavra, freq in ranking:
    barra = "" * int(freq)
    print(f"  {palavra:>12} | {barra:<10} {freq}")


FREQUÊNCIA GLOBAL DAS PALAVRAS (todos os docs)
      cachorro |            4
          gato |            4
          casa |            2
       brincam |            1
         corre |            1
         dorme |            1
        juntos |            1
          late |            1
           rua |            1
           tem |            1


In [8]:
# ─────────────────────────────────────────────
# 6. SIMILARIDADE DE COSSENO
# ─────────────────────────────────────────────

def similaridade_cosseno(v1: np.ndarray, v2: np.ndarray) -> float:
    """
    Mede o quão parecidos são dois documentos em direção vetorial.
    Retorna valor entre 0 (nenhuma similaridade) e 1 (idênticos).
    """
    norma = np.linalg.norm(v1) * np.linalg.norm(v2)
    if norma == 0:
        return 0.0
    return float(np.dot(v1, v2) / norma)

print("\n" + "=" * 55)
print("SIMILARIDADE DE COSSENO ENTRE DOCUMENTOS")
print("=" * 55)

n = len(corpus)
print(f"\n{'':>8}", end="")
for i in range(n):
    print(f"  Doc {i}", end="")
print()

for i in range(n):
    print(f"Doc {i}  ", end="")
    for j in range(n):
        sim = similaridade_cosseno(bow[i], bow[j])
        print(f"  {sim:.2f} ", end="")
    print()

# Par mais similar (excluindo diagonal)
max_sim = -1
par_similar = (0, 1)
for i in range(n):
    for j in range(i + 1, n):
        sim = similaridade_cosseno(bow[i], bow[j])
        if sim > max_sim:
            max_sim = sim
            par_similar = (i, j)

i, j = par_similar
print(f"\nPar mais similar: Doc {i} e Doc {j} (cosseno = {max_sim:.2f})")
print(f"  Doc {i}: \"{corpus[i]}\"")
print(f"  Doc {j}: \"{corpus[j]}\"")


SIMILARIDADE DE COSSENO ENTRE DOCUMENTOS

          Doc 0  Doc 1  Doc 2  Doc 3  Doc 4
Doc 0    1.00   0.33   0.33   0.58   0.58 
Doc 1    0.33   1.00   0.00   0.29   0.58 
Doc 2    0.33   0.00   1.00   0.29   0.29 
Doc 3    0.58   0.29   0.29   1.00   0.50 
Doc 4    0.58   0.58   0.29   0.50   1.00 

Par mais similar: Doc 0 e Doc 3 (cosseno = 0.58)
  Doc 0: "O cachorro late para o gato"
  Doc 3: "O gato e o cachorro brincam juntos"


In [9]:
# ─────────────────────────────────────────────
# 7. CONSULTA: NOVO DOCUMENTO
# ─────────────────────────────────────────────

print("\n" + "=" * 55)
print("CONSULTA — NOVO DOCUMENTO")
print("=" * 55)

novo_doc = "cachorro late na rua"
tokens_novo = preprocessar(novo_doc)
vetor_novo = np.zeros(len(vocab), dtype=np.int32)

for palavra in tokens_novo:
    if palavra in word2idx:
        vetor_novo[word2idx[palavra]] += 1

print(f"\nConsulta: \"{novo_doc}\"")
print(f"Tokens:   {tokens_novo}")
print(f"Vetor:    {vetor_novo}")

print("\nSimilaridade com cada documento do corpus:")
sims = [(i, similaridade_cosseno(vetor_novo, bow[i])) for i in range(n)]
sims.sort(key=lambda x: -x[1])

for i, sim in sims:
    barra = "" * int(sim * 20)
    print(f"  Doc {i}: {sim:.2f} {barra}  \"{corpus[i]}\"")


CONSULTA — NOVO DOCUMENTO

Consulta: "cachorro late na rua"
Tokens:   ['cachorro', 'late', 'rua']
Vetor:    [0 1 0 0 0 0 0 1 1 0]

Similaridade com cada documento do corpus:
  Doc 0: 0.67   "O cachorro late para o gato"
  Doc 2: 0.67   "O cachorro corre na rua"
  Doc 3: 0.29   "O gato e o cachorro brincam juntos"
  Doc 4: 0.29   "A casa tem um cachorro e um gato"
  Doc 1: 0.00   "O gato dorme na casa"
